# Aula 12 — Regra da cadeia aplicada à rede

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/04-deep-learning/m5-redes-neurais-do-zero/notebooks/12-regra-cadeia-rede-laboratorio.ipynb)

Laboratório reproduzível em **NumPy puro** para executar forward e backward de uma rede escalar completa. O arquivo versionado não contém outputs; execute as células em ordem.

## Objetivos e ambiente

- Conectar BCE, camada afim, tanh e camada afim em ordem reversa.
- Rastrear todos os intermediários e gradientes escalares.
- Confirmar parâmetros e entradas por diferenças centrais.
- Testar direção de descida, acumulação, cache e saturação.

Dependências mínimas: Python >= 3.11, NumPy >= 1.26, Matplotlib >= 3.8 e nbformat >= 5.9 para validação. Seed fixa: `20260912`.

Não há downloads, credenciais, frameworks de deep learning ou aleatoriedade sem seed.

In [ ]:
from importlib.metadata import version
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260912
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=9, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("nbformat:", version("nbformat"))
print("Seed:", SEED)

## 1. Rede escalar e contratos

Implementaremos

$$z=w_1x_1+w_2x_2+b,\quad a=\tanh(z),\quad o=va+c,$$

seguido de BCE estável em logits. Todos os parâmetros são escalares finitos; $x$ tem exatamente duas posições e $y\in[0,1]$.

In [ ]:
PARAMETER_NAMES = ("w1", "w2", "b", "v", "c")


def finite_scalar(name, value):
    result = np.asarray(value, dtype=np.float64)
    if result.shape != () or not np.isfinite(result):
        raise ValueError(f"{name} deve ser escalar finito")
    return float(result)


def validate_inputs(params, x, y):
    if set(params) != set(PARAMETER_NAMES):
        raise ValueError(f"parâmetros devem ser {PARAMETER_NAMES}")
    clean_params = {name: finite_scalar(name, params[name]) for name in PARAMETER_NAMES}
    x = np.asarray(x, dtype=np.float64)
    if x.shape != (2,) or not np.all(np.isfinite(x)):
        raise ValueError("x deve ter shape (2,) e valores finitos")
    y = finite_scalar("y", y)
    if not 0.0 <= y <= 1.0:
        raise ValueError("y deve pertencer a [0, 1]")
    return clean_params, x.copy(), y


def sigmoid_scalar(value):
    value = finite_scalar("logit", value)
    if value >= 0:
        return 1.0 / (1.0 + np.exp(-value))
    exp_value = np.exp(value)
    return exp_value / (1.0 + exp_value)

## 2. Forward com cache imutável

A BCE é calculada como `logaddexp(0, o) - y*o`. O cache guarda cópias dos valores que produziram a loss; o backward não consulta parâmetros externos mutáveis.

In [ ]:
def network_forward(params, x, y):
    p, x, y = validate_inputs(params, x, y)
    z = p["w1"] * x[0] + p["w2"] * x[1] + p["b"]
    a = float(np.tanh(z))
    o = p["v"] * a + p["c"]
    probability = sigmoid_scalar(o)
    loss = float(np.logaddexp(0.0, o) - y * o)
    cache = {
        "params": p.copy(), "x": x.copy(), "y": y,
        "z": z, "a": a, "o": o, "probability": probability,
    }
    return loss, cache


params = {"w1": 0.8, "w2": -0.4, "b": 0.1, "v": -1.2, "c": 0.3}
x = np.array([1.5, -0.5])
y = 1.0
loss, cache = network_forward(params, x, y)

for name in ("z", "a", "o", "probability"):
    print(f"{name:>11}: {cache[name]: .9f}")
print(f"{'loss':>11}: {loss: .9f}")

assert np.isclose(cache["z"], 1.5)
assert np.isclose(cache["a"], np.tanh(1.5))
assert 0.0 < cache["probability"] < 0.5
assert np.isfinite(loss)

## 3. Backward completo

Começamos em $\bar L=1$. A BCE fundida fornece $\bar o=p-y$; depois passamos pela camada de saída, pela tanh e pela camada oculta.

In [ ]:
def network_backward(cache, upstream=1.0):
    upstream = finite_scalar("upstream", upstream)
    p = cache["params"]
    x_local = cache["x"]
    a = cache["a"]
    probability = cache["probability"]
    y_local = cache["y"]

    dloss = upstream
    do = dloss * (probability - y_local)
    dv = do * a
    dc = do
    da = do * p["v"]
    dz = da * (1.0 - a**2)
    dw1 = dz * x_local[0]
    dw2 = dz * x_local[1]
    db = dz
    dx = np.array([dz * p["w1"], dz * p["w2"]])

    parameter_grads = {"w1": dw1, "w2": dw2, "b": db, "v": dv, "c": dc}
    adjoints = {"L": dloss, "o": do, "a": da, "z": dz}
    return parameter_grads, dx, adjoints


grads, dx, adjoints = network_backward(cache)
print("Adjuntos:")
for name, value in adjoints.items():
    print(f"  {name:>2} barra = {value: .9f}")
print("Gradientes de parâmetros:")
for name in PARAMETER_NAMES:
    print(f"  d{name:>2} = {grads[name]: .9f}")
print("Gradientes das entradas:", dx)

assert adjoints["L"] == 1.0
assert np.isclose(adjoints["o"], cache["probability"] - y)
assert set(grads) == set(PARAMETER_NAMES)
assert dx.shape == x.shape

## 4. Produto por caminho

Expandimos os gradientes de $w_1$ e $x_1$ como produtos das derivadas locais. Isso confirma que o algoritmo reutiliza o mesmo prefixo, sem mudar a matemática.

In [ ]:
do_direct = cache["probability"] - y
dw1_path = do_direct * params["v"] * (1.0 - cache["a"]**2) * x[0]
dx1_path = do_direct * params["v"] * (1.0 - cache["a"]**2) * params["w1"]
prefix_z = do_direct * params["v"] * (1.0 - cache["a"]**2)

print("prefixo até z:", prefix_z)
print("dw1 por caminho:", dw1_path)
print("dx1 por caminho:", dx1_path)

assert np.isclose(prefix_z, adjoints["z"])
assert np.isclose(dw1_path, grads["w1"])
assert np.isclose(dx1_path, dx[0])

## 5. Gradient checking coordenado

Cada perturbação reconstrói o forward. Verificaremos os cinco parâmetros e as duas entradas em `float64`.

In [ ]:
def relative_error(analytic, numeric):
    return abs(analytic - numeric) / max(1.0, abs(analytic), abs(numeric))


def parameter_numeric_gradient(params, x, y, step=1e-6):
    result = {}
    for name in PARAMETER_NAMES:
        plus, minus = params.copy(), params.copy()
        plus[name] += step
        minus[name] -= step
        result[name] = (
            network_forward(plus, x, y)[0] - network_forward(minus, x, y)[0]
        ) / (2.0 * step)
    return result


def input_numeric_gradient(params, x, y, step=1e-6):
    result = np.empty_like(x, dtype=np.float64)
    for index in range(x.size):
        plus, minus = x.copy(), x.copy()
        plus[index] += step
        minus[index] -= step
        result[index] = (
            network_forward(params, plus, y)[0] - network_forward(params, minus, y)[0]
        ) / (2.0 * step)
    return result


numeric_params = parameter_numeric_gradient(params, x, y)
numeric_x = input_numeric_gradient(params, x, y)
parameter_errors = {name: relative_error(grads[name], numeric_params[name]) for name in PARAMETER_NAMES}
input_errors = np.array([relative_error(dx[i], numeric_x[i]) for i in range(2)])

for name in PARAMETER_NAMES:
    print(f"{name:>2}: analítico={grads[name]: .9f} numérico={numeric_params[name]: .9f} erro={parameter_errors[name]:.3e}")
print("erros das entradas:", input_errors)

assert max(parameter_errors.values()) < 3e-10
assert np.max(input_errors) < 3e-10

## 6. Teste direcional do vetor de parâmetros

Empacotamos os cinco parâmetros em um vetor apenas para verificar $\nabla L^\top d$. A rede continua escalar; a vetorização da MLP fica para a próxima aula.

In [ ]:
def pack(mapping):
    return np.array([mapping[name] for name in PARAMETER_NAMES], dtype=np.float64)


def unpack(vector):
    vector = np.asarray(vector, dtype=np.float64)
    if vector.shape != (len(PARAMETER_NAMES),):
        raise ValueError("vetor de parâmetros com shape inválido")
    return dict(zip(PARAMETER_NAMES, vector))


theta = pack(params)
gradient_vector = pack(grads)
direction = rng.normal(size=theta.shape)
direction /= np.linalg.norm(direction)
step = 1e-6
numeric_directional = (
    network_forward(unpack(theta + step * direction), x, y)[0]
    - network_forward(unpack(theta - step * direction), x, y)[0]
) / (2.0 * step)
analytic_directional = float(gradient_vector @ direction)
directional_error = relative_error(analytic_directional, numeric_directional)

print("direcional analítica:", analytic_directional)
print("direcional numérica:", numeric_directional)
print("erro relativo:", directional_error)
assert directional_error < 3e-10

## 7. Backpropagation versus atualização

O backward termina antes de qualquer mutação. Depois, aplicamos $\theta\leftarrow\theta-\eta g$. Para um passo pequeno, a loss deve cair aproximadamente $\eta\lVert g\rVert^2$.

In [ ]:
learning_rate = 0.1
updated_theta = theta - learning_rate * gradient_vector
updated_params = unpack(updated_theta)
updated_loss, _ = network_forward(updated_params, x, y)
linear_prediction = loss - learning_rate * float(gradient_vector @ gradient_vector)

print("loss original:", loss)
print("loss após um passo:", updated_loss)
print("aproximação linear:", linear_prediction)
print("redução observada:", loss - updated_loss)

assert updated_loss < loss
assert abs(updated_loss - linear_prediction) < 0.02
assert all(params[name] != updated_params[name] for name in PARAMETER_NAMES)

## 8. Contraprova: omitir a derivada da tanh

Um backward incorreto pode manter shapes e sinais. Remover $1-a^2$ superestima todos os gradientes anteriores à ativação.

In [ ]:
wrong_dz = adjoints["a"]
wrong_grads_hidden = {
    "w1": wrong_dz * x[0],
    "w2": wrong_dz * x[1],
    "b": wrong_dz,
}
inflation = abs(wrong_grads_hidden["w1"] / grads["w1"])
wrong_error = relative_error(wrong_grads_hidden["w1"], numeric_params["w1"])

print("fator correto da tanh:", 1.0 - cache["a"]**2)
print("inflação ao omitir o fator:", inflation)
print("erro relativo do dw1 incorreto:", wrong_error)

assert np.isclose(inflation, 1.0 / (1.0 - cache["a"]**2))
assert inflation > 5.0
assert wrong_error > 0.5

## 9. Ramificação e acumulação

O parâmetro compartilhado $r$ aparece em $o=ra+rx+c$. Seu gradiente deve somar as duas rotas.

In [ ]:
def shared_loss(r, a, x_scalar, c, y):
    o = r * a + r * x_scalar + c
    return float(np.logaddexp(0.0, o) - y * o)


r, a_shared, x_shared, c_shared, y_shared = 0.7, -0.4, 1.2, 0.1, 1.0
o_shared = r * a_shared + r * x_shared + c_shared
do_shared = sigmoid_scalar(o_shared) - y_shared
path_a = do_shared * a_shared
path_x = do_shared * x_shared
dr_accumulated = path_a + path_x
dr_numeric = (
    shared_loss(r + 1e-6, a_shared, x_shared, c_shared, y_shared)
    - shared_loss(r - 1e-6, a_shared, x_shared, c_shared, y_shared)
) / 2e-6

print("caminho via a:", path_a)
print("caminho via x:", path_x)
print("soma:", dr_accumulated)
print("numérico:", dr_numeric)

assert relative_error(dr_accumulated, dr_numeric) < 1e-10
assert relative_error(path_x, dr_numeric) > 0.1

## 10. Cache antigo não representa parâmetros novos

O cache preserva corretamente o forward original. Depois de mudar parâmetros, é obrigatório executar novo forward antes do novo backward.

In [ ]:
stale_grads, _, _ = network_backward(cache)
new_params = params.copy()
new_params["v"] = 0.9
new_params["b"] = -0.7
new_loss, new_cache = network_forward(new_params, x, y)
fresh_grads, _, _ = network_backward(new_cache)
new_numeric = parameter_numeric_gradient(new_params, x, y)

fresh_error = max(relative_error(fresh_grads[n], new_numeric[n]) for n in PARAMETER_NAMES)
stale_error = max(relative_error(stale_grads[n], new_numeric[n]) for n in PARAMETER_NAMES)
print("loss no novo estado:", new_loss)
print("erro com cache novo:", fresh_error)
print("erro com cache antigo:", stale_error)

assert fresh_error < 3e-10
assert stale_error > 0.1
assert params["v"] == cache["params"]["v"] == -1.2

## 11. Saturação localizada

Fixamos upstream $\bar a=1$ e comparamos o adjunto $\bar z=1-a^2$ em $z=0$ e $z=6$. Assim isolamos a ativação do restante da rede.

In [ ]:
z_values = np.array([0.0, 1.5, 6.0])
a_values = np.tanh(z_values)
dz_values = 1.0 - a_values**2
attenuation = dz_values[0] / dz_values[-1]

for z_value, a_value, dz_value in zip(z_values, a_values, dz_values):
    print(f"z={z_value:>3.1f} tanh(z)={a_value:.9f} fator={dz_value:.9e}")
print("atenuação centro / z=6:", attenuation)

assert dz_values[0] == 1.0
assert dz_values[-1] < 2.5e-5
assert attenuation > 40000

## 12. Aproximação local e direção de descida

Visualizamos a loss em $\theta-tg$. Em $t=0$, a inclinação deve ser negativa; longe da origem, a aproximação linear deixa de ser exata.

In [ ]:
t_values = np.linspace(-0.35, 0.55, 181)
path_losses = np.array([
    network_forward(unpack(theta - t * gradient_vector), x, y)[0]
    for t in t_values
])
linear_losses = loss - t_values * float(gradient_vector @ gradient_vector)
small_t = 1e-4
small_actual = network_forward(unpack(theta - small_t * gradient_vector), x, y)[0]
small_linear = loss - small_t * float(gradient_vector @ gradient_vector)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(t_values, path_losses, label="loss real")
ax.plot(t_values, linear_losses, "--", label="aproximação local")
ax.axvline(0.0, color="black", linewidth=0.8)
ax.scatter([0.0], [loss], color="black", zorder=3, label="estado inicial")
ax.set(xlabel="t em θ - t∇L", ylabel="BCE", title="Loss ao longo da direção de descida")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

print("erro da aproximação em t=1e-4:", abs(small_actual - small_linear))
assert small_actual < loss
assert abs(small_actual - small_linear) < 1e-7

**Texto alternativo do gráfico:** a curva da BCE real e sua reta tangente são mostradas em função de `t` no caminho `θ - t∇L`. Ambas partem da loss inicial em `t=0`; para pequenos valores positivos, a loss diminui, enquanto a aproximação linear se afasta gradualmente da curva real.

## 13. Falhas antecipadas

Contratos rejeitam label fora do intervalo, entrada com shape incorreto, parâmetro ausente, upstream vetorial e valor não finito.

In [ ]:
invalid_calls = [
    lambda: network_forward(params, np.ones(3), y),
    lambda: network_forward(params, x, 2.0),
    lambda: network_forward({"w1": 1.0}, x, y),
    lambda: network_forward({**params, "b": np.inf}, x, y),
    lambda: network_backward(cache, np.ones(2)),
]
contract_messages = []
for call in invalid_calls:
    try:
        call()
    except ValueError as exc:
        contract_messages.append(str(exc))
    else:
        raise AssertionError("entrada inválida deveria falhar")

assert len(contract_messages) == 5
for message in contract_messages:
    print("Contrato acionado:", message)

## 14. Auditoria final

Os grupos abaixo consolidam forward, backward, caminhos, verificações numéricas, atualização, ramificação, cache, saturação e contratos.

In [ ]:
audit = {
    "forward finito": np.isfinite(loss),
    "pré-ativação manual": np.isclose(cache["z"], 1.5),
    "BCE inicia backward": np.isclose(adjoints["o"], cache["probability"] - y),
    "shapes das entradas": dx.shape == x.shape,
    "produto por caminho": np.isclose(dw1_path, grads["w1"]),
    "parâmetros por diferenças centrais": max(parameter_errors.values()) < 3e-10,
    "entradas por diferenças centrais": np.max(input_errors) < 3e-10,
    "teste direcional": directional_error < 3e-10,
    "passo reduz loss": updated_loss < loss,
    "derivada omitida detectada": wrong_error > 0.5,
    "ramificação acumulada": relative_error(dr_accumulated, dr_numeric) < 1e-10,
    "cache novo coerente": fresh_error < 3e-10,
    "cache antigo detectado": stale_error > 0.1,
    "saturação mensurada": attenuation > 40000,
    "contratos": len(contract_messages) == 5,
}

for name, passed in audit.items():
    assert passed, name
print(f"Auditoria concluída: {sum(audit.values())}/{len(audit)} grupos aprovados.")

## Conclusões

- O backward começou em $\bar L=1$ e percorreu BCE, saída afim, tanh e afim oculta.
- Todos os parâmetros e entradas coincidiram com diferenças centrais.
- O teste direcional validou o vetor completo de parâmetros.
- Um passo pequeno em $-\nabla L$ reduziu a loss.
- Derivada omitida, contribuição sobrescrita e cache antigo foram detectados.

Na Aula 13, a mesma lógica será vetorizada para lotes e matrizes em uma MLP de duas camadas.